## Step 1: Fine-type Generation (Few-shot)

In [ ]:
## Code for input data generation (i)
import json
from tqdm import tqdm

FEW_SHOT_EXAMPLES = [
    {
        "entity": "G-Dragon",
        "type": "PER",
        "context": "Can't believe I finally saw G-Dragon live in Seoul last night! 🔥 #Kpop",
        "fine_type": "K-pop idol"
    },
    {
        "entity": "Serena Williams",
        "type": "PER",
        "context": "Serena Williams just crushed it at Wimbledon again 👏 #GOAT",
        "fine_type": "professional tennis player"
    },
    {
        "entity": "Steve Jobs",
        "type": "PER",
        "context": "Rewatching Steve Jobs' 2007 iPhone keynote. Absolute game-changer. 📱",
        "fine_type": "tech entrepreneur"
    },
    {
        "entity": "FC Barcelona",
        "type": "OTHER",
        "context": "What a goal by FC Barcelona tonight! 🔵🔴 #ForçaBarça",
        "fine_type": "professional football club"
    },
    {
        "entity": "Google",
        "type": "organization",
        "context": "Google just announced a new AI update at I/O, this is huge 🤯",
        "fine_type": "tech company"
    },
    {
        "entity": "Namsan Tower",
        "type": "location",
        "context": "Sunset from Namsan Tower is unreal... 🌇 #SeoulVibes",
        "fine_type": "landmark / tourist attraction"
    }
]


def build_user_prompt(entity_name: str, coarse_type: str, input_text: str) -> str:
    prompt = "[Instruction]\nBelow are examples of inferring fine-grained entity types from given coarse entity types and their original texts.\n\n"
    for ex in FEW_SHOT_EXAMPLES:
        prompt += f"Entity: {ex['entity']}\nType: {ex['type']}\nText: {ex['context']}\nFine-grained Type: {ex['fine_type']}\n\n"
    prompt += f"[Target]\nEntity: {entity_name}\nType: {coarse_type}\nText: {input_text}\nFine-grained Type:"
    return prompt

def make_batch_jsonl(input_json_path: str, output_jsonl_path: str):
    """
    input_json_path: Raw JSON
    output_jsonl_path: JSONL for GPT batch
    """
    with open(input_json_path, "r", encoding="utf-8") as infile, \
         open(output_jsonl_path, "w", encoding="utf-8") as outfile:

        for idx, line in enumerate(tqdm(infile, desc="Building batch input")):
            data = json.loads(line)
            
            input_text = data.get("text", "")
            for ent_idx, ent in enumerate(data.get("entities", [])):
                entity_name = ent["text"]
                entity_type = ent["type"] # assume type always exist

                payload = {
                    "custom_id": f"{data['image']}_ent{ent_idx}",
                    "method": "POST",
                    "url": "/v1/chat/completions",
                    "body": {
                        "model": "gpt-4o-mini",
                        "temperature": 0.2,
                        "max_tokens": 64,
                        "messages": [
                            {"role": "system",
                             "content": "You are an expert assistant that infers fine-grained types of named entities, "
                                        "based on the entity name, its coarse type, and the original text the entity was in. "
                                        "Be concise and use realistic categories."
                                        "Coarse types are defined as : "
                                        "- PER = person "
                                        "- ORG = organization"
                                        "- LOC = location"
                                        "- OTHER = miscellaneous (e.g. sports teams, awards, TVshows, etc.)"
                            },
                            {"role": "user", "content": build_user_prompt(entity_name, entity_type, input_text)}
                        ]
                    }
                }
                outfile.write(json.dumps(payload, ensure_ascii=False) + "\n")




if __name__ == "__main__":
    input_json = r""
    output_jsonl = r""
    make_batch_jsonl(input_json, output_jsonl)
    print(f"✅ Created batch input: {output_jsonl}")


In [ ]:
## Code for creating GPT Batch (ii)

from openai import OpenAI
client = OpenAI(api_key=API_KEY)
# available only in version after openai==1.2.0
batch_input_file = client.files.create(
  file=open(r"", "rb"),
  purpose="batch"
)

batch_input_file_id = batch_input_file.id

client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h", 
    metadata={
      "description": "entity fine type generation"
    }
)

In [ ]:
## Code for output file geneartion (iii)
 
import json
from collections import defaultdict

input_path = r""
output_path = r""
final_path = r""           

# custom_id -> inferred_fine_type mapping
fine_type_map = defaultdict(dict)

with open(output_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        custom_id = obj["custom_id"]
        content = obj["response"]["body"]["choices"][0]["message"]["content"]

        # custom_id example: "797970.jpg_ent0"
        image_name, ent_part = custom_id.split("_ent")
        ent_idx = int(ent_part)

        fine_type_map[image_name][ent_idx] = content

# adding inferred_fine_type to the input file
new_lines = []

with open(input_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        image = obj["image"]

        if image in fine_type_map:
            for idx, entity in enumerate(obj["entities"]):
                if idx in fine_type_map[image]:
                    entity["inferred_fine_type"] = fine_type_map[image][idx]

        new_lines.append(obj)

with open(final_path, "w", encoding="utf-8") as f:
    for obj in new_lines:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print(f"Done! Created the final file: {final_path}")


## Step 2: Discriminative Visual Knoweldge Generation

In [ ]:
## Code for input data generation (i)
import json
from tqdm import tqdm

def build_user_prompt(entity_name: str, entity_type: str, input_text: str):
    return (
        "You are an assistant helping a visual recognition system to locate a specific entity in an image.\n\n"
        f"Entity: {entity_name}\n"
        f"Fine-grained entity type: {entity_type}\n"
        f"Original Text: \"{input_text}\"\n"
        "The system has absolutely no background knowledge and can only rely on visible visual features that you say.\n"
        f"Using your prior visual knowledge of the entity and the provided information, write a single-sentence description "
        f"that would help the system find {entity_name} from other {entity_type}(s/es) in the image.\n"
        "Focus only on visible characteristics such as appearance, colors, or shape. "
        "Avoid abstract terms, affiliations, or concepts that cannot be seen."
    )

def make_batch_jsonl(input_json_path: str, output_jsonl_path: str):
    """
    input_json_path: Raw JSON 
    output_jsonl_path: JSONL for GPT batch
    """
    with open(input_json_path, "r", encoding="utf-8") as infile, \
         open(output_jsonl_path, "w", encoding="utf-8") as outfile:

        for idx, line in enumerate(tqdm(infile, desc="Building batch input")):
            data = json.loads(line)

            input_text = data.get("text", "")
            for ent_idx, ent in enumerate(data.get("entities", [])):
                entity_name = ent["text"]
                entity_type = ent["inferred_fine_type"]  # ✅ assume types always exist

                payload = {
                    "custom_id": f"{data['image']}_ent{ent_idx}",
                    "method": "POST",
                    "url": "/v1/chat/completions",
                    "body": {
                        "model": "gpt-4o-mini",
                        "temperature": 0.2,
                        "max_tokens": 64,
                        "messages": [
                            {"role": "user", "content": build_user_prompt(entity_name, entity_type, input_text)}
                        ]
                    }
                }
                outfile.write(json.dumps(payload, ensure_ascii=False) + "\n")



if __name__ == "__main__":
    input_json = r""  
    output_jsonl = r""
    make_batch_jsonl(input_json, output_jsonl)
    print(f"✅ Created batch input: {output_jsonl}")


In [ ]:
## Code for creating GPT Batch (ii)

from openai import OpenAI
client = OpenAI(api_key=API_KEY)
# available only in version after openai==1.2.0
batch_input_file = client.files.create(
  file=open(r"", "rb"),
  purpose="batch"
)

batch_input_file_id = batch_input_file.id

client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h", 
    metadata={
      "description": "entity knowledge generation"
    }
)

In [ ]:
## Code for output file geneartion (iii)
import json
from collections import defaultdict

input_path = r""  
output_path = r"" # batch output file
final_path = r"" # file to save 

# custom_id -> visual_knowledge mapping
visual_map = defaultdict(dict)

with open(output_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        custom_id = obj["custom_id"]
        content = obj["response"]["body"]["choices"][0]["message"]["content"]

        # custom_id: "1768609.jpg_ent0" -> image = "1768609.jpg", ent_idx = 0
        image_name, ent_part = custom_id.split("_ent")
        ent_idx = int(ent_part)

        visual_map[image_name][ent_idx] = content

# add visual_knowledge
new_lines = []

with open(input_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        # print(obj)
        image = obj["image"]

        if image in visual_map:
            for idx, entity in enumerate(obj["entities"]):
                if idx in visual_map[image]:
                    entity["entity_knowledge"] = visual_map[image][idx]

        new_lines.append(obj)

with open(final_path, "w", encoding="utf-8") as f:
    for obj in new_lines:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print(f"Done! Created the final output: {final_path}")

In [ ]:
## Code for input data generation (i)
import json
from tqdm import tqdm

SYSTEM_PROMPT = """You are a data preprocessing assistant for a vision-language model.
        Your task is to rewrite entity descriptions so that they only include **visually observable features**.

        Rules:
        - Keep only attributes that can be recognized visually in an image:
        - Physical appearance (gender, approximate age, hairstyle, facial hair, body type)
        - Clothing, uniforms, or accessories (colors, styles, numbers, logos)
        - Distinguishing objects, actions, or poses
        - Scene or landmark characteristics (for locations)
        - Logo colors or shapes (for organizations)
        - Remove non-visual information:
        - Biographical details, achievements, dates, numbers of wins
        - Personality traits, abstract facts, historical or textual information
        - Anything that cannot be seen in a single image
        - Keep the output concise (under 30 words), formatted as a **short comma-separated phrase list**.
        - Do NOT add extra explanations or full sentences.
        - If the entity has almost no visual attributes, just return the entity name.
        - Make it short! into a few keywords or phrases. Only the most visually distinctive features.

        Example:
        Input: "Clint Dempsey is an American professional soccer player, usually wearing the number 2 jersey for Seattle Sounders, with short brown hair and a lean build."
        Output: "male soccer player, short brown hair, number 2 jersey, lean build"
        Example: 
        Input: "The Seattle Sounders are a Major League Soccer team based in Seattle, known for their green and blue logo and large fan banners at home games."
        Output: "green and blue team logo, soccer team uniforms, large stadium banners"
        Example: 
        Input: "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, standing 330 meters tall and illuminated at night."
        Output: "tall iron lattice tower, triangular shape, illuminated at night"
"""

def build_user_prompt(entity_name: str, entity_knowledge: str):
    return f"""
Rewrite this entity description very short to include only visually observable features:

Entity: {entity_name}
Original Description: {entity_knowledge}

Output:
"""

def make_batch_jsonl(input_json_path: str, output_jsonl_path: str):

    with open(input_json_path, "r", encoding="utf-8") as infile, \
         open(output_jsonl_path, "w", encoding="utf-8") as outfile:

        for idx, line in enumerate(tqdm(infile, desc="Building batch input")):
            data = json.loads(line)
            for ent_idx, ent in enumerate(data.get("entities", [])):
                entity_name = ent["text"]
                entity_knowledge = ent.get("entity_knowledge", "")
                # print(ent_id, ent)
                # entity_knowledge 없으면 pass
                # if not entity_knowledge:
                #     continue

                payload = {
                    "custom_id": f"{data['image']}_ent{ent_idx}",
                    "method": "POST",
                    "url": "/v1/chat/completions",
                    "body": {
                        "model": "gpt-4o",
                        "temperature": 0.2,
                        "max_tokens": 64,
                        "messages": [
                            {"role": "system", "content": SYSTEM_PROMPT},
                            {"role": "user", "content": build_user_prompt(entity_name, entity_knowledge)}
                        ]
                    }
                }
                outfile.write(json.dumps(payload, ensure_ascii=False) + "\n")


if __name__ == "__main__":
    input_json = r""  
    output_jsonl = r""
    make_batch_jsonl(input_json, output_jsonl)
    print(f"✅ Created batch input: {output_jsonl}")


In [ ]:
## Code for creating GPT Batch (ii)

from openai import OpenAI
client = OpenAI(api_key="")
# available only in version after openai==1.2.0
batch_input_file = client.files.create(
  file=open(r"", "rb"),
  purpose="batch"
)

batch_input_file_id = batch_input_file.id

client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h", 
    metadata={
      "description": "visual knowledge filteringE"
    }
)

In [ ]:
## Code for output file geneartion (iii)
import json
from collections import defaultdict

input_path = r""
output_path = r""
final_path = r""

# 1. custom_id -> visual_knowledge mapping
visual_map = defaultdict(dict)

with open(output_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        custom_id = obj["custom_id"]
        content = obj["response"]["body"]["choices"][0]["message"]["content"]

        # custom_id: "1768609.jpg_ent0" -> image = "1768609.jpg", ent_idx = 0
        image_name, ent_part = custom_id.split("_ent")
        ent_idx = int(ent_part)

        visual_map[image_name][ent_idx] = content

# 2. add visual_knowledge to the original file
new_lines = []

with open(input_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        # print(obj)
        image = obj["image"]

        if image in visual_map:
            for idx, entity in enumerate(obj["entities"]):
                if idx in visual_map[image]:
                    # entity["visual_knowledge"] = visual_map[image][idx].replace('\"', '')
                    try: 
                        entity["visual_knowledge"] = json.loads(visual_map[image][idx])
                    except:
                        entity["visual_knowledge"] = visual_map[image][idx]

        new_lines.append(obj)


with open(final_path, "w", encoding="utf-8") as f:
    for obj in new_lines:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print(f"Done! Saved the output file: {final_path}")